# Day 3 · A review threshold you can defend

A model gives scores, but a review team needs a rule: *flag this request or not*. Today we compare three ways of handling the rare default class, then pick a threshold using a teaching loss (`10 x FN + FP`) and a review capacity of 12% of requests per validation period.

## 1. Setup

In [1]:
from pathlib import Path
import hashlib
import importlib.util
import os
import runpy
import signal
import subprocess
import sys
import time
import urllib.error
import urllib.request

RUN_STARTED = time.perf_counter()

COURSE_REVISION = "fe0c0204e6076a7ac2139b7336485a097343fb8a"
MANIFEST_SHA256 = "86d15e813caf833b7d86512808176388d4123dc3c01c6525ee849a76e8322c59"
SETUP_SHA256 = "79608f8f9559d9e6eb09d51ee541ee12d7e9262ac57d1ca3a948c8ecb14f0f35"
RAW_BASE = "https://raw.githubusercontent.com/almiyead-rgb/sda-dsc-211-student-template"


def running_in_colab():
    try:
        return importlib.util.find_spec("google.colab") is not None
    except ModuleNotFoundError:
        return False


IN_COLAB = running_in_colab()
ROOT = Path("/content/tamweel") if IN_COLAB else Path(os.environ.get("TAMWEEL_PROJECT_DIR", ".")).resolve()
ROOT.mkdir(parents=True, exist_ok=True)

setup_path = ROOT / "scripts" / "setup_colab.py"
setup_path.parent.mkdir(parents=True, exist_ok=True)
if not setup_path.exists() or hashlib.sha256(setup_path.read_bytes()).hexdigest() != SETUP_SHA256:
    for attempt in range(3):
        try:
            with urllib.request.urlopen(f"{RAW_BASE}/{COURSE_REVISION}/scripts/setup_colab.py", timeout=45) as reply:
                payload = reply.read(200_000)
            if hashlib.sha256(payload).hexdigest() != SETUP_SHA256:
                raise ValueError("Setup checksum mismatch. Reopen the released notebook.")
            setup_path.write_bytes(payload)
            break
        except (urllib.error.URLError, TimeoutError):
            if attempt == 2:
                raise RuntimeError("Download unavailable. Retry, or upload the unchanged project files.")
            time.sleep(attempt + 1)

prepare = runpy.run_path(str(setup_path))["prepare"]
try:
    environment = prepare(ROOT, COURSE_REVISION, MANIFEST_SHA256)
except RuntimeError as problem:
    if "already imported" not in str(problem):
        raise

    print("Installing the pinned package versions (one time, 1-2 minutes)...", flush=True)
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--quiet",
         "-r", str(ROOT / "requirements-colab.txt"), "-c", str(ROOT / "constraints.txt")],
        capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError("Package installation failed:\n" + result.stderr[-3000:])
    print("Done. The runtime restarts by itself now (a 'session crashed' notice is expected).\n"
          "After it restarts, choose Runtime > Run all again.", flush=True)
    time.sleep(2)
    os.kill(os.getpid(), signal.SIGKILL)

FAST_MODE, FULL_MODE, SEED, N_JOBS = True, False, 211, 2

from setup_colab import fetch_verified

DAY2_REVISION = "1e1da4acbee8941bef07245b259fb6c27ced4ad7"
DAY2_SHA256 = "d2fd0353330c3629bdaed6c570528792955fce232540a85d33532a9432522f43"
fetch_verified(f"{RAW_BASE}/{DAY2_REVISION}/scripts/day2_validation.py",
               ROOT / "scripts/day2_validation.py", DAY2_SHA256)

LAB_REVISION = "c2dc52e129878d0d487db1874891c658de7c4d0d"
SUPPORT_HASHES = {
    "scripts/day3_decision.py": "40f176633c91fc530db222978bc44495a401b888b1ce448728948f7ae0ca9e12",
    "data/day3_oof_example.csv": "659952bef85ac3b7ceb7ab84eb2102de6c9b433a1ba122cd57df96581baf0eb3",
    "data/day3_oof_example.json": "920cf9830043afcc83f8c276c8945677296eb3c49a6e69326d47f6080a9e7fad",
}
for relative, digest in SUPPORT_HASHES.items():
    fetch_verified(f"{RAW_BASE}/{LAB_REVISION}/{relative}", ROOT / relative, digest)
print("Day 3 setup verified")

Package                 Version
numpy                   2.1.3
pandas                  2.2.3
scipy                   1.16.3
scikit-learn            1.6.1
matplotlib              3.10.0
xgboost                 3.4.1
lightgbm                4.6.0
shap                    0.52.0
numba                   0.61.2
optuna                  4.5.0
Python 3.13.18 | CPU | seed=211 | n_jobs=2 | FAST_MODE=True
Verified course files. Next: run the data and compatibility checks.
Day 3 setup verified


## 2. Fix the policy before looking at results

The target is a default within 90 days of applying. Two rules are frozen up front: a missed default (FN) costs 10 teaching units and a false alarm (FP) costs 1, and no validation period may flag more than 12% of its requests. Folds are the same later-period folds as Day 2, so training only sees mature labels from other customers.

In [2]:
import json
import os
import platform
import zipfile
from dataclasses import asdict, replace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, roc_curve
from data_checks import validate_frame
from day2_validation import deduplicate_applications, make_time_group_split
from day3_decision import (
    STRATEGIES, TARGET, CostPolicy, DecisionConfig, TrainingBudgetExpired,
    flag_at, generate_oof, load_example, period_audit, reflection_check,
    region_audit, select_threshold, threshold_sweep, validate_oof
)

contract = json.loads(
    (ROOT / "data/data_contract.json").read_text(encoding="utf-8")
)

train, duplicates = deduplicate_applications(
    pd.read_csv(ROOT / "data/tamweel_train.csv")
)

validate_frame(train, contract, "train")

config = DecisionConfig(trees=80 if FAST_MODE else 160)
policy = CostPolicy(**contract["decision_policy"])

STRATEGY_FOR_DECISION = "weighted"
USE_EXAMPLE = False
ASSESSMENT_MODE = os.environ.get("ASSESSMENT_MODE", "0") == "1"

assert STRATEGY_FOR_DECISION in STRATEGIES

ARTIFACTS, REPORTS = ROOT / "artifacts", ROOT / "reports"
ARTIFACTS.mkdir(exist_ok=True)
REPORTS.mkdir(exist_ok=True)

folds = make_time_group_split(
    train,
    horizon_days=contract["target"]["horizon_days"]
)

summary = pd.DataFrame([{
    "training_rows": len(train),
    "positive_rate": train[TARGET].mean(),
    "FN_units": policy.false_negative_cost,
    "FP_units": policy.false_positive_cost,
    "capacity_per_period": policy.max_flag_fraction,
    "trees_per_model": config.trees
}])

display(summary)

fold_audit = pd.DataFrame([
    fold["audit"] for fold in folds
])

display(fold_audit)

  training_rows positive_rate  FN_units  FP_units  capacity_per_period  trees_per_model
0         10000        0.0789        10         1                 0.12               80

   fold validation_start validation_end_exclusive  train_rows  validation_rows  train_positives  validation_positives  immature_past_rows_removed  customer_overlap_rows_removed  shared_customers latest_training_label_available
0     1       2023-07-01             2024-01-01        3223             1632              274                   110                           826                             912                 0                      2023-06-30
1     2       2024-01-01             2024-07-01        4460             1674              353                   135                           785                            1348                 0                      2023-12-31
2     3       2024-07-01             2025-01-01        5731             1733              465                   139                           861                            1675                 0                      2024-06-30

## 3. Change only the training, never the validation

Three LightGBM strategies share the same features, folds, tree budget and seed. They differ only in how the rare default class is treated during training: **unweighted**, **class-weighted**, **oversampled**. Validation rows are never reweighted or resampled.

In [3]:
from pathlib import Path
import hashlib
import importlib.util
import os
import runpy
import signal
import subprocess
import sys
import time
import urllib.error
import urllib.request

RUN_STARTED = time.perf_counter()

COURSE_REVISION = "fe0c0204e6076a7ac2139b7336485a097343fb8a"
MANIFEST_SHA256 = "86d15e813caf833b7d86512808176388d4123dc3c01c6525ee849a76e8322c59"
SETUP_SHA256 = "79608f8f9559d9e6eb09d51ee541ee12d7e9262ac57d1ca3a948c8ecb14f0f35"
RAW_BASE = "https://raw.githubusercontent.com/almiyead-rgb/sda-dsc-211-student-template"


def running_in_colab():
    try:
        return importlib.util.find_spec("google.colab") is not None
    except ModuleNotFoundError:
        return False


IN_COLAB = running_in_colab()
ROOT = Path("/content/tamweel") if IN_COLAB else Path(os.environ.get("TAMWEEL_PROJECT_DIR", ".")).resolve()
ROOT.mkdir(parents=True, exist_ok=True)

setup_path = ROOT / "scripts" / "setup_colab.py"
setup_path.parent.mkdir(parents=True, exist_ok=True)
if not setup_path.exists() or hashlib.sha256(setup_path.read_bytes()).hexdigest() != SETUP_SHA256:
    for attempt in range(3):
        try:
            with urllib.request.urlopen(f"{RAW_BASE}/{COURSE_REVISION}/scripts/setup_colab.py", timeout=45) as reply:
                payload = reply.read(200_000)
            if hashlib.sha256(payload).hexdigest() != SETUP_SHA256:
                raise ValueError("Setup checksum mismatch. Reopen the released notebook.")
            setup_path.write_bytes(payload)
            break
        except (urllib.error.URLError, TimeoutError):
            if attempt == 2:
                raise RuntimeError("Download unavailable. Retry, or upload the unchanged project files.")
            time.sleep(attempt + 1)

prepare = runpy.run_path(str(setup_path))["prepare"]
try:
    environment = prepare(ROOT, COURSE_REVISION, MANIFEST_SHA256)
except RuntimeError as problem:
    if "already imported" not in str(problem):
        raise

    print("Installing the pinned package versions (one time, 1-2 minutes)...", flush=True)
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--quiet",
         "-r", str(ROOT / "requirements-colab.txt"), "-c", str(ROOT / "constraints.txt")],
        capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError("Package installation failed:\n" + result.stderr[-3000:])
    print("Done. The runtime restarts by itself now (a 'session crashed' notice is expected).\n"
          "After it restarts, choose Runtime > Run all again.", flush=True)
    time.sleep(2)
    os.kill(os.getpid(), signal.SIGKILL)

FAST_MODE, FULL_MODE, SEED, N_JOBS = True, False, 211, 2

from setup_colab import fetch_verified

DAY2_REVISION = "1e1da4acbee8941bef07245b259fb6c27ced4ad7"
DAY2_SHA256 = "d2fd0353330c3629bdaed6c570528792955fce232540a85d33532a9432522f43"
fetch_verified(f"{RAW_BASE}/{DAY2_REVISION}/scripts/day2_validation.py",
               ROOT / "scripts/day2_validation.py", DAY2_SHA256)

LAB_REVISION = "c2dc52e129878d0d487db1874891c658de7c4d0d"
SUPPORT_HASHES = {
    "scripts/day3_decision.py": "40f176633c91fc530db222978bc44495a401b888b1ce448728948f7ae0ca9e12",
    "data/day3_oof_example.csv": "659952bef85ac3b7ceb7ab84eb2102de6c9b433a1ba122cd57df96581baf0eb3",
    "data/day3_oof_example.json": "920cf9830043afcc83f8c276c8945677296eb3c49a6e69326d47f6080a9e7fad",
}
for relative, digest in SUPPORT_HASHES.items():
    fetch_verified(f"{RAW_BASE}/{LAB_REVISION}/{relative}", ROOT / relative, digest)
print("Day 3 setup verified")

LIVE: your own CPU run


   eligible_rows  all_rows  whole_data_coverage  eligible_coverage  warmup_without_oof source
0           5039     10000               0.5039                1.0                4961   LIVE

      strategy  fold  original_training_rows  fitted_rows  scale_pos_weight  training_positive_rate  fitted_positive_rate  average_precision  train_seconds source
0   unweighted     1                    3223         3223            1.0000                  0.0850                0.0850             0.3395         0.4349   LIVE
1     weighted     1                    3223         3223           10.7628                  0.0850                0.0850             0.3378         0.2318   LIVE
2  oversampled     1                    3223         5898            1.0000                  0.0850                0.5000             0.3502         1.1966   LIVE
3   unweighted     2                    4460         4460            1.0000                  0.0791                0.0791             0.3403         1.8444   LIVE
4     weighted     2                    4460         4460           11.6346